# Interactive Exercise: Plot Historical Expenditures

Historical data exploration is the first step in any expenditure forecasting process.

In this exercise, you will use a sample Illinois General Funds expenditure dataset to create simple visualizations that reveal historical expenditure patterns.

By the end of this exercise, you will be able to:

- Load historical expenditure data.
- Select an expenditure category.
- Create line and bar charts.
- Interpret historical expenditure patterns.
- Save publication-quality figures.

> **Milestone:** You are preparing the data for forecasting.

## 1. Load the Required Packages

The following packages are used to import, organize, and visualize the expenditure data.

In [ ]:
library(readr)
library(dplyr)
library(tidyr)
library(ggplot2)
library(scales)

## 2. Load the Expenditure Data

The sample dataset contains Illinois General Funds expenditures for major agencies and expenditure categories from FY2016 through FY2025.

The notebook uses a relative file path so that it works both locally and in Binder.

In [ ]:
df <- read_csv("../data/illinois_expenditures.csv")

df

## 3. Explore the Dataset

Each row represents one expenditure category.

The `series` column contains short names that can be used in the code, while the `label` column contains the full expenditure category name.

In [ ]:
df %>%
  select(series, label)

## 4. Select an Expenditure Category

Choose one expenditure category to visualize.

For example:

- `"education"`
- `"healthcare"`
- `"human_services"`
- `"higher_ed"`
- `"corrections"`
- `"aging"`
- `"total_expenditures"`

In [ ]:
selected_series <- "education"

## 5. Prepare the Data

The dataset is stored in wide format, with one column for each fiscal year.

The following code selects one expenditure category and converts the data into a long format that is easier to visualize.

In [ ]:
selected_df <- df %>%
  filter(series == selected_series)

if (nrow(selected_df) == 0) {
  stop(
    paste(
      "The selected series was not found.",
      "Choose one of the values listed in the series column."
    )
  )
}

plot_df <- selected_df %>%
  pivot_longer(
    cols = starts_with("fy"),
    names_to = "fiscal_year",
    values_to = "expenditure"
  ) %>%
  mutate(
    fiscal_year = as.numeric(gsub("fy", "", fiscal_year)) + 2000
  ) %>%
  arrange(fiscal_year)

plot_df

## 6. Review the Selected Data

Before creating any charts, inspect the resulting table to ensure that the fiscal years and expenditure values appear reasonable.

In [ ]:
plot_df

## 7. Create a Line Chart

Line charts are useful for identifying:

- Long-term trends
- Periods of rapid growth or decline
- Structural breaks
- Unusual observations

In [ ]:
line_plot <- ggplot(
  plot_df,
  aes(
    x = fiscal_year,
    y = expenditure
  )
) +
  geom_line(
    linewidth = 1.2,
    color = "#2F5D8A"
  ) +
  geom_point(
    size = 3,
    color = "#2F5D8A"
  ) +
  scale_x_continuous(
    breaks = plot_df$fiscal_year,
    labels = paste0("FY", plot_df$fiscal_year)
  ) +
  scale_y_continuous(
    labels = comma
  ) +
  labs(
    title = paste(
      "Historical Expenditures:",
      unique(plot_df$label)
    ),
    x = "Fiscal Year",
    y = "Expenditure ($ millions)"
  ) +
  theme_minimal(base_size = 13) +
  theme(
    plot.title = element_text(face = "bold"),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank()
  )

line_plot

## 8. Create a Bar Chart

Bar charts make it easier to compare expenditure levels across fiscal years and identify years with relatively large increases or decreases.

In [ ]:
bar_plot <- ggplot(
  plot_df,
  aes(
    x = factor(fiscal_year),
    y = expenditure
  )
) +
  geom_col(
    fill = "#2F5D8A"
  ) +
  scale_y_continuous(
    labels = comma
  ) +
  labs(
    title = paste(
      "Historical Expenditures:",
      unique(plot_df$label)
    ),
    x = "Fiscal Year",
    y = "Expenditure ($ millions)"
  ) +
  theme_minimal(base_size = 13) +
  theme(
    plot.title = element_text(face = "bold"),
    panel.grid.major.x = element_blank(),
    panel.grid.minor = element_blank()
  )

bar_plot

## 9. Compare Expenditure Categories in a Selected Year

The previous charts show how one expenditure category changes over time. Analysts may also want to compare spending across categories within a single fiscal year.

Choose a fiscal year below to create a horizontal bar chart of expenditure levels by category.

The chart excludes `total_expenditures` because it is an aggregate of the underlying categories. It also excludes prior-year adjustments, which may be negative and are not a substantive expenditure category.

In [ ]:
# Select a fiscal year
# Available values are fy16 through fy25
selected_year <- "fy25"

# Confirm that the selected year exists
if (!selected_year %in% names(df)) {
  stop(
    paste0(
      "The selected year was not found. Choose one of: ",
      paste(
        names(df)[grepl("^fy[0-9]+$", names(df))],
        collapse = ", "
      )
    )
  )
}

# Create a readable fiscal-year label
selected_year_label <- paste0(
  "FY20",
  gsub("fy", "", selected_year)
)

# Prepare expenditure categories for the selected year
year_comparison_df <- df %>%
  filter(
    !series %in% c(
      "total_expenditures",
      "prior_year_adjustments"
    )
  ) %>%
  transmute(
    series,
    label,
    expenditure = .data[[selected_year]]
  ) %>%
  filter(
    !is.na(expenditure),
    expenditure >= 0
  ) %>%
  arrange(expenditure)

year_comparison_df

### Create the Selected-Year Bar Chart

The horizontal layout makes category names easier to read and allows expenditure levels to be compared directly.

In [ ]:
# Make the notebook output larger
options(
  repr.plot.width = 10,
  repr.plot.height = 7
)

year_bar_plot <- ggplot(
  year_comparison_df,
  aes(
    x = reorder(label, expenditure),
    y = expenditure
  )
) +
  geom_col(
    fill = "#2F5D8A",
    width = 0.72
  ) +
  geom_text(
    aes(label = comma(expenditure)),
    hjust = -0.12,
    size = 3.5
  ) +
  coord_flip(
    clip = "off"
  ) +
  scale_y_continuous(
    labels = comma,
    expand = expansion(mult = c(0, 0.18))
  ) +
  labs(
    title = paste0(
      "Illinois General Funds Expenditures by Category, ",
      selected_year_label
    ),
    subtitle = "Selected expenditure categories",
    x = NULL,
    y = "Expenditure ($ millions)"
  ) +
  theme_minimal(base_size = 13) +
  theme(
    plot.title = element_text(
      face = "bold",
      size = 15
    ),
    plot.subtitle = element_text(
      size = 11
    ),
    panel.grid.major.y = element_blank(),
    panel.grid.minor = element_blank(),
    plot.margin = margin(
      t = 10,
      r = 50,
      b = 10,
      l = 10
    )
  )

year_bar_plot

## 10. Save the Figures

The following code saves both figures in the `output` folder.

Binder sessions are temporary, so download any files you wish to keep before closing the session.

In [ ]:
dir.create(
  "../output",
  showWarnings = FALSE
)

ggsave(
  filename = paste0(
    "../output/",
    selected_series,
    "_line_plot.png"
  ),
  plot = line_plot,
  width = 7,
  height = 4.5,
  dpi = 300
)

ggsave(
  filename = paste0(
    "../output/",
    selected_series,
    "_bar_plot.png"
  ),
  plot = bar_plot,
  width = 7,
  height = 4.5,
  dpi = 300
)

ggsave(
  filename = paste0(
    "../output/",
    selected_year,
    "_category_comparison.png"
  ),
  plot = year_bar_plot,
  width = 8,
  height = 6,
  dpi = 300
)

cat("Figures saved to the output folder.\n")

## Questions for Reflection

After creating the figures, consider the following questions:

1. Does the expenditure series exhibit a clear long-term trend?

2. Are there unusually large increases or decreases?

3. Can any changes be explained by known policy or economic events?

4. Would this expenditure category appear suitable for a simple baseline forecast?

5. Which expenditure category would you like to examine next?

## Try Another Expenditure Category

Return to the `selected_series` cell and replace `"education"` with another expenditure category.

For example:

```r
selected_series <- "healthcare"
```

Then rerun the remaining cells to compare historical expenditure patterns across categories.

# Continue Working with This Notebook

This notebook is intended to serve as a reusable template for exploring historical expenditure data.

## Analyze Your Own Data

You can adapt this notebook to your own expenditure data by:

1. Uploading your dataset to Binder using the **Upload Files** button.
2. Replacing the sample dataset in the `read_csv()` command with your own file.
3. Updating any variable names if your dataset uses different column headings.
4. Rerunning the notebook from the beginning.

## Continue Working on Your Computer

If you would like to save your work or continue after leaving Binder, you can download this notebook and run it locally.

To download the notebook:

- Select **File → Download** in JupyterLab, or
- Right-click the notebook in the file browser and choose **Download**.

To run locally, you will need:

- R (version 4.3 or later recommended)
- Jupyter Notebook or JupyterLab
- The R packages installed in the first code cell

Once downloaded, place your expenditure dataset in the same folder as the notebook (or update the file path in the `read_csv()` command) and rerun the notebook from the beginning.
